# Анаис — Оптимизированная версия (Минимальная память 4135 MiB + Ускоренный запуск)
**Decentrathon 6.0: Qwen2.5-1.5B-Instruct на Tesla T4**

- **Память:** Снижена с **7,040 MiB до ~4,135 MiB** (экономия почти **3 ГБ** VRAM) за счёт устранения утечки кэша слоёв NF4.
- **Время:** **~44.0 с** (screening 43.91–44.02 с) благодаря lazy CLI (`lazy_train_cli`), убирающему импорт лишних модулей Soup.
- **Правила:** 10/10 PASS, Eval loss 2.0496 (строго равен Base), 19 собственных Triton-ядер (доля > 5%).

In [ ]:
import subprocess, sys
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True, check=True)
print(gpu.stdout.strip())
assert 'Tesla T4' in gpu.stdout, 'Execution requires NVIDIA Tesla T4 GPU'


In [ ]:
# Decentrathon 6.0: Optimized Anais (v10 / balanced speed & low-memory)
# Peak memory: ~4,135 MiB (down from 7,040 MiB, saving ~2.9 GB VRAM)
# Whole command time: ~44s (screening ~43.9-44.0s) with lazy CLI & unpadded tokens
# Evaluated on Kaggle T4 x2 / Internet On. Paired runs, 10/10 checks.

import base64, io, zipfile, pathlib, subprocess, sys, json, shutil

project = pathlib.Path('/kaggle/working/anais_optimized')
if project.exists():
    shutil.rmtree(project)
project.mkdir(parents=True)

PAYLOAD = '...'
with zipfile.ZipFile(io.BytesIO(base64.b64decode(PAYLOAD))) as archive:
    archive.extractall(project)

# Reuse prepared data if available, otherwise prepare
data_candidates = [
    pathlib.Path('/kaggle/working/anais_v7/data'),
    pathlib.Path('/kaggle/working/anais_fastest_v6/anais-evaluation/data')
]
reused = False
for c in data_candidates:
    if c.exists():
        shutil.copytree(c, project / 'data')
        reused = True
        break
if not reused:
    subprocess.run([sys.executable, 'task/prepare_data.py'], cwd=project, check=True)

# Run solution tests
subprocess.run([sys.executable, '-m', 'pytest', 'solution', '-q'], cwd=project, check=True)

# Run full task measurement (2 cold paired repeats)
subprocess.run([sys.executable, 'harness/run_task.py', '--repeats', '2'], cwd=project, check=True)

# Verify all 10 rules
checks = subprocess.run([sys.executable, 'harness/check_rules.py'], cwd=project)

result_file = project / 'runs/result.json'
if result_file.exists():
    print('\n=== OPTIMIZATION SUMMARY ===')
    print(json.dumps(json.loads(result_file.read_text())['summary'], indent=2))

assert checks.returncode == 0, 'Rules verification failed! Inspect runs/checks.json'
